> 📄 English: [mem0_tutorial.ipynb](mem0_tutorial.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial)

# 使用 Mem0 为 AI Agent 构建持久化记忆

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NirDiamant/agents-towards-production/blob/main/tutorials/agent-memory-with-mem0/mem0_tutorial.ipynb)

## 🎯 简介

AI agent 面临一个根本性局限：**记忆失忆症**。它们在每次对话后忘记一切，无法从过往交互中学习，并丢失了本可以让它们成为真正智能助手的宝贵上下文。

虽然基础记忆方案存储的是静态信息，但 **[Mem0](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=mem0-nir&target=https%3A%2F%2Fmem0.dev%2Fgithub%2Fnir&text=Mem0)** 引入了一种革命性的方法：**自我改进的记忆**，它能自动提取洞察、解决冲突，并随着每次交互不断演化。Mem0 不只是存储对话，而是构建一个智能知识系统，学习用户偏好并提供上下文理解。

## 🛠️ 我们要构建什么

我们将创建一个智能的「个人 AI 研究助手」，展示 Mem0 的完整能力：

它会：

*   跟踪你对深度、风格和格式的偏好
*   将研究笔记、摘要和洞察存储到向量记忆中，用于语义召回
*   从你的笔记中捕获关键想法，并在图数据库中链接相关概念
*   识别新问题是否与之前的讨论相关，并浮现正确的上下文
*   同时使用相似度搜索和结构化关系，随时间生成更有依据的回答

这个助手不必每次会话都从零开始，而是持续在其已有知识之上构建——帮助你形成对研究主题不断增长、相互关联的理解。

## 🏗️ 记忆架构概览

**Mem0** 是面向 AI agent 的记忆层，它能自动从对话中提取重要信息，将其存储到向量和图数据库中，并在之后检索，从而提供长期上下文和连续性。

下图是展示 Mem0 如何处理和存储记忆的高层流程。

![mem0_architecture](Assets/mem0_architecture.png)

## 🎢 学习旅程

**阶段 1：向量记忆基础**

*   基础助手：用向量记忆构建一个可用的研究助手
*   记忆操作：亲身体验知识提取和冲突解决
*   语义搜索：体验超越关键词的智能检索
*   自我改进：观察记忆随着每次交互不断演化

**阶段 2：图增强**

*   图能力：添加实体关系映射
*   研究网络：跟踪论文引用和作者关联
*   知识图谱：可视化研究领域的关系

## ⚙️ 前置条件与环境配置

本教程需要：

1. [OpenAI API Key](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=openai-platform&target=https%3A%2F%2Fplatform.openai.com%2Fsignup&text=OpenAI%20API%20Key) - 用于 LLM 处理（GPT-4）和嵌入
2. [Qdrant Cloud Account](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=qdrant&target=https%3A%2F%2Fcloud.qdrant.io&text=Qdrant%20Cloud%20Account) 或 Mem0 支持的任何其他向量存储，用于向量记忆。
3. [Neo4j AuraDB](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=neo4j-aura&target=https%3A%2F%2Fneo4j.com%2Faura%2F&text=Neo4j%20AuraDB) 或 Mem0 支持的任何其他图数据库，用于图记忆

### 安装

让我们先安装所有必需的依赖：

In [ ]:
!pip install -q mem0ai openai python-dotenv neo4j

### 导入所需库

In [ ]:
# Core libraries
import os
import getpass

# Mem0 and OpenAI
from mem0 import Memory
from openai import OpenAI

### 所需的 API 密钥

让我们配置你的 API 密钥：

In [ ]:
def setup_api_keys():

    # OpenAI API Key
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

    # Qdrant Cloud (Required for Phase 1: Vector Memory)

    if "QDRANT_URL" not in os.environ:
        os.environ["QDRANT_URL"] = input("\nEnter your Qdrant Cloud URL: ")
    if "QDRANT_API_KEY" not in os.environ:
        os.environ["QDRANT_API_KEY"] = getpass.getpass("Enter your Qdrant API key: ")

    return True

# Run the setup
use_qdrant_cloud = setup_api_keys()

## 记忆配置
## 阶段 1：向量记忆

现在让我们配置 Mem0 的记忆系统。我们从**基于向量的记忆**开始——这是让其他一切成为可能的基础。


### 理解配置

每个组件在记忆流水线中都有特定的用途：

- **🤖 LLM Provider**：提取洞察并解决冲突
- **📊 Embedder**：将文本转换为语义向量
- **🗄️ Vector Store**：存储嵌入以进行相似度搜索

In [ ]:
if use_qdrant_cloud:
    # Enhanced configuration with Qdrant Cloud
    config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "gpt-4o-mini",
            "temperature": 0.1,
            "max_tokens": 2000,
        }
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-large",
            "embedding_dims": 3072,
        }
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "url": os.environ["QDRANT_URL"],
            "api_key": os.environ["QDRANT_API_KEY"],
            "collection_name": "research_assistant_vectors",
            "embedding_model_dims": 3072,  # Must match embedding model
        }
    },
    "version": "v1.1",  # Latest version with enhanced features
}

print("Vector Memory Configuration Created!")

### 初始化记忆系统

现在让我们创建我们的记忆实例——研究助手的大脑：

In [ ]:
try:
    memory = Memory.from_config(config)
    print("Memory system initialized successfully!")
except Exception as e:
    print(f"Error initializing memory: {e}")

## 构建研究助手

现在，我们将使用 Mem0 的向量记忆能力构建我们的个人研究助手。

### Mem0 的核心操作

在构建助手之前，先了解 Mem0 的主要操作：

`memory.add()` - 存储新信息
- 自动从对话中提取关键事实
- 处理去重和冲突
- 按用户隔离存储记忆

![mem0_add_architecture](Assets/mem0_add_architecture.jpg)

`memory.search()` - 检索相关记忆
- 查找语义相似的内容（不只是关键词匹配）
- 按相关性返回排序结果
- 支持按 user_id 过滤

![mem0_search_architecture](Assets/mem0_search_architecture.jpg)

**其他操作**（本教程未使用）：
- `memory.update()` - 修改现有记忆
- `memory.delete()` - 删除特定记忆

In [ ]:
class PersonalResearchAssistant:

    def __init__(self, memory_instance):
        self.client = OpenAI()
        self.memory = memory_instance  # Use the configured memory instance
        print(f"Research Assistant initialized with Mem0 memory!")

    def ask(self, question, user_id):

        # Search for relevant memories
        previous_memories = self.search_memories(question, user_id=user_id)

        # Build the prompt with memory context
        system_message = "You are a personal AI Research Assistant. Help users with research questions, remember their interests, and provide contextual recommendations."

        if previous_memories:
            memory_context = ", ".join(previous_memories)
            prompt = f"{system_message}\n\nUser input: {question}\nPrevious memories: {memory_context}"
        else:
            prompt = f"{system_message}\n\nUser input: {question}"

        # Generate response
        try:
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": prompt},
                ],
                temperature=0.1,
                max_tokens=2000
            )

            answer = response.choices[0].message.content

            # Store the question in memory
            self.memory.add(question, user_id=user_id, metadata={"category": "research"})

            return answer

        except Exception as e:
            return f"Encountered an error: {e}"

    def get_memories(self, user_id):
        try:
            memories = self.memory.get_all(user_id=user_id)
            if isinstance(memories, dict) and 'results' in memories:
                return [m['memory'] for m in memories['results']]
            elif isinstance(memories, list):
                return [m['memory'] for m in memories]
            return []
        except Exception as e:
            print(f"Error retrieving memories: {e}")
            return []

    def search_memories(self, query, user_id):
        try:
            memories = self.memory.search(query, user_id=user_id)
            if isinstance(memories, dict) and 'results' in memories:
                return [m['memory'] for m in memories['results']]
            elif isinstance(memories, list):
                return [m['memory'] for m in memories]
            return []
        except Exception as e:
            print(f"Error searching memories: {e}")
            return []

# Initialize our research assistant
assistant = PersonalResearchAssistant(memory)
print("\n Research Assistant ready to learn and help!")

### 测试 1：首次交互 - 知识提取

让我们从一个基础的研究兴趣开始：

In [ ]:
response1 = assistant.ask(
    "I'm interested in transformer architectures for natural language processing. "
    "Can you help me find recent papers on this topic?",
    user_id="researcher"
)

print(f"Assistant: {response1}")

### 测试 2：构建上下文 - 观察记忆如何连接想法

In [ ]:
response2 = assistant.ask(
    "I prefer papers that include practical implementation details and code examples. "
    "Theoretical papers without code are less useful for my work.",
    user_id="researcher"
)

print(f"Assistant: {response2}")

### 测试 3：语义搜索 - 超越关键词的智能

In [ ]:
response3 = assistant.ask(
    "What about BERT and GPT models? Are they related to my research interests?",
    user_id="researcher"
)

print(f"Assistant: {response3}")

### 测试 4：冲突解决 - 智能记忆管理

In [ ]:
response4 = assistant.ask(
    "Actually, I also need to understand the theoretical foundations of attention mechanisms. "
    "Can you recommend some foundational theory papers?",
    user_id="researcher"
)

print(f"Assistant: {response4}")

## 📊 记忆分析 - 看看它学到了什么

让我们检查一下助手从这些交互中学到了什么：

In [ ]:
def analyze_extracted_memories():

    print("📈 MEMORY ANALYSIS - What Did the Assistant Learn?")
    print("=" * 60)

    # Get all memories
    all_memories = assistant.get_memories(user_id="researcher")

    if all_memories:
        print(f"\n🧠 Total memories extracted: {len(all_memories)}")
        print(f"\n📚 Key insights Mem0 learned about you:")

        for i, memory in enumerate(all_memories, 1):
            print(f"\n{i}. {memory}")

        # Test semantic search
        print(f"\n🔍 Testing Semantic Search Capabilities:")

        test_queries = [
            "neural networks",           # Should connect to transformers
            "code implementations",      # Should find practical preferences
            "attention mechanisms",      # Should connect to transformer interest
            "deep learning papers"       # Should find research interests
        ]

        for query in test_queries:
            related_memories = assistant.search_memories(query, user_id="researcher")
            print(f"\n   🔎 Query: '{query}'")
            print(f"      Found {len(related_memories)} related memories")
            if related_memories:
                print(f"      Top match: {related_memories[0][:100]}...")

    else:
        print("\n⚠️ No memories found. Try running the interaction tests first.")

    return all_memories

# Run the analysis
memories = analyze_extracted_memories()

## 交互式演示 - 亲自试试

现在轮到你了！试着提问，观察记忆系统如何学习你的研究兴趣：

In [ ]:
def interactive_demo():
    while True:
        try:
            question = input("\nYou: ")

            if question.lower() in ['quit', 'exit', 'q']:
                print("\n Thanks for trying the Research Assistant!")
                break
            elif question.lower() == 'memories':
                memories = assistant.get_memories(user_id="researcher")
                if memories:
                    print(f"\n Here's what I've learned about your research interests:")
                    for i, memory in enumerate(memories, 1):
                        print(f"   {i}. {memory}")
                else:
                    print("\n No memories yet. Start asking about your research interests!")
            elif question.strip():
                response = assistant.ask(question)
                print(f"\n Assistant: {response}")

        except KeyboardInterrupt:
            print("\n\n Demo ended. Thanks for trying the Research Assistant!")
            break
        except Exception as e:
            print(f"\n Error: {e}")

# Run the interactive demo
interactive_demo()

## 🚀 阶段 2：用图记忆能力增强

既然你已经掌握了**基于向量的记忆**并见识了它的威力，接下来让我们用**图能力**增强系统。这一新增能力将实现实体、概念和研究领域之间的显式关系映射。

### 为什么要添加图记忆？

**向量记忆擅长**：

- 查找语义相似的内容
- 「给我推荐像这样的论文」
- 理解概念上的相似性

**图记忆额外提供**：

- 显式的实体关系
- 「谁和谁合作过？」
- 「论文 A 是如何影响论文 B 的？」
- 研究谱系和引用网络

### 图存储配置

要添加图能力，我们需要一个图数据库。Neo4j 是最流行的选择，你也可以使用 Mem0 支持的其他图存储。

In [ ]:
def setup_graph_capabilities():

    print("\n🔗 Enter your Neo4j connection details:")
    neo4j_uri = input("Neo4j URI (e.g., neo4j+s://xxx.databases.neo4j.io): ")
    neo4j_username = input("Username (usually 'neo4j'): ")
    neo4j_password = getpass.getpass("Password: ")

    print("\n Neo4j configuration complete!")

    return True, {
        "url": neo4j_uri,
        "username": neo4j_username,
        "password": neo4j_password
    }

# Setup graph capabilities for Phase 2
has_graph, graph_config = setup_graph_capabilities()

### 使用图存储的增强配置

现在让我们创建同时包含向量存储和图存储的混合配置：

In [ ]:
# Enhanced configuration with both vector and graph storage
enhanced_config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "gpt-4o-mini",
            "temperature": 0.1,
            "max_tokens": 2000,
        }
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "text-embedding-3-large",
            "embedding_dims": 3072,
        }
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "url": os.environ["QDRANT_URL"],
            "api_key": os.environ["QDRANT_API_KEY"],
            "collection_name": "research_assistant_hybrid",
            "embedding_model_dims": 3072,
        }
    },
    "graph_store": {
        "provider": "neo4j",
        "config": graph_config
    },
    "version": "v1.1",
}

try:
    enhanced_memory = Memory.from_config(enhanced_config)
    print("\n Hybrid Memory System Initialized Successfully!")

    enhanced_assistant = PersonalResearchAssistant(enhanced_memory)
    print("\n Enhanced Research Assistant ready with hybrid memory!")

except Exception as e:
    print(f" Error initializing hybrid memory: {e}")

    # Fallback to vector-only
    print("\n Falling back to vector-only assistant...")
    enhanced_assistant = assistant

### 📝 图增强研究示例

如果你已启用图存储，让我们通过实体关系看看它的实际效果：

### 示例 1：研究谱系映射

In [ ]:
graph_response1 = enhanced_assistant.ask(
    "I'm studying the lineage of transformer papers. The original 'Attention Is All You Need' "
    "by Vaswani et al. led to BERT by Devlin et al., and then to many other models. "
    "Can you help me map these research connections and suggest related work?",
    user_id="graph_user"
)

print(f"Hybrid Assistant: {graph_response1}")

### 示例 2：合作网络分析

In [ ]:
graph_response2 = enhanced_assistant.ask(
    "What other researchers have worked on transformer architectures? "
    "I want to understand the collaboration network and research groups in this field.",
    user_id="graph_user"
)

print(f"Hybrid Assistant: {graph_response2}")

## 🎓 回顾

你刚刚从零构建了一个真正的记忆系统。以下是你的成果以及后续方向。

### 你构建了什么
- **混合记忆架构**，包含向量搜索（Qdrant）+ 图关系（Neo4j）
- **自动记忆提取**，从对话中提取关键事实
- **冲突解决**，处理矛盾并让记忆保持最新
- **语义搜索**，按含义而非仅按关键词查找相关信息

### 为什么选择 Mem0

记忆很难做好。要正确处理事实提取、冲突解决和语义搜索，你可能需要几个月时间。Mem0 为你提供了久经考验、开箱即用的记忆能力。

**你能得到：**
- 智能记忆提取，自动从对话中提取关键事实
- 当用户改变主意或自相矛盾时的冲突处理
- 按含义（而非仅关键词）查找相关内容的语义搜索
- 经过验证的性能：准确率提升 26%、响应快 91%、token 减少 90%（Mem0 研究论文）

你可以专注于构建应用，而不是重新发明记忆基础设施。

### 你的部署选项

**自托管（我们刚才构建的方式）**
- 完全掌控你的技术栈
- 向量/图存储的自定义配置
- 非常适合学习、原型开发和特定需求

**Mem0 Platform**
- 托管服务，免费档位起步（1 万条记忆）
- 分析和图记忆等企业级特性
- 简单的 API，零基础设施烦恼
- 快速上线并扩展，无需运维负担


### 下一步
你现在已经具备了让 AI 真正记住并学习的基础。请查看：

- **[Mem0 Platform](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=mem0-nir&target=https%3A%2F%2Fmem0.dev%2Fdashboard%2Fnir&text=Mem0%20Platform)** 用于生产部署
- **[Documentation](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=mem0-nir&target=https%3A%2F%2Fmem0.dev%2Fdocs%2Fnir&text=Documentation)** 用于高级特性和集成
- **[Discord Community](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-memory-with-mem0--mem0-tutorial&click=mem0-nir&target=https%3A%2F%2Fmem0.dev%2Fdiscord%2Fnir&text=Discord%20Community)** 用于提问和讨论

你已经准备好构建随着每次交互变得越来越聪明的 AI 应用了。